In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import json
from typing import Dict, List, Any

In [0]:
from pyspark.sql import DataFrame
def f_llm_lookup(
    table_path: str,
    input_reference_columns: list,
    output_reference_columns: str,
    endpoint_name: str,
    query_text: str,
):
    try:
        import json
        import pandas as pd

        df = spark.read.table(table_path).toPandas()

        selected_columns = input_reference_columns 
        output_column = output_reference_columns  # Get the single column name
        all_columns = set([output_column] + selected_columns)
        formatted_data = df[all_columns].to_csv(index=False)

        prompt = f"""# Data Matching Assistant Prompt

**You are a data matching assistant.**

Your task is to identify the best matching row from a reference dataset based on a given input query, and return the matching row as a json.

## Instructions:
- You will receive:
  - A query text
  - A reference dataset in CSV format containing the columns - "{'","'.join(all_columns)}"
  - Use **semantic similarity, fuzzy matching, and contextual understanding** to find the best row match, even if the query is an abbreviation, partial name, or variant.
  - **Match should be based on all columns** in the reference data, but return only the value from the specified output column.
  - Return only the JSON version of the matching row. Do not include any other text or formatting.
  - If no good match is found, return the original query text as-is.

---

**Query Text:** `"{query_text}"`
**Reference Data (CSV):**
\"\"\"
{formatted_data}
\"\"\""""
        print(prompt)
        response = spark.sql(
            f"""
        SELECT ai_query('lakefusion-{endpoint_name}', '{prompt.replace("'", "''")}') AS response
        """
        ).collect()[0]["response"]

        return response
    except Exception as e:
        raise Exception(f"Error in value mapper transformation: {str(e)}")

import json
from pyspark.sql.types import *
def f_llm_lookup_execute(
    input_df: DataFrame,
    refernce_table: str,
    input_reference_columns: list,
    output_reference_columns: str,
    endpoint_name: str,
    lookup_column: str,
    new_column_name: str,
):
    result_df = []
    list_columns = input_df.columns
    
    for row in input_df.collect():
        row_dict = row.asDict()
        row_values = [row_dict[col] for col in list_columns]
        query_val = row_dict.get(lookup_column, "")
        matched = f_llm_lookup(
            table_path=refernce_table,
            input_reference_columns=input_reference_columns,
            output_reference_columns=output_reference_columns,
            endpoint_name=endpoint_name,
            query_text=query_val,
        )
        data_output = json.loads(matched).get(output_reference_columns,"")
        result_df.append(row_values + [data_output])
        
   
    schema = StructType([*(StructField(col, StringType()) for col in list_columns),StructField(new_column_name,(StringType()))])  
    final_df=spark.createDataFrame(result_df, schema=schema)
    return final_df
    

In [0]:
# from pyspark.sql import DataFrame
# def f_llm_lookup(
#     table_path: str,
#     input_reference_columns: list,
#     output_reference_columns: str,
#     endpoint_name: str,
#     query_text: str,
# ):
#     try:
#         import json
#         import pandas as pd

#         df = spark.read.table(table_path).toPandas()

#         selected_columns = input_reference_columns 
#         output_column = output_reference_columns  # Get the single column name
#         all_columns = selected_columns + [output_column]
#         formatted_data = df[all_columns].to_csv(index=False)

#         prompt = f"""
#        You are a precise data matching assistant

# Your ONLY task is to identify the best matching row from a reference dataset based on a given input query, and return ONLY the value from the specified output column.

# ## CRITICAL INSTRUCTIONS:
# 1. You will receive:
#    - A query text
#    - A reference dataset in CSV format
#    - The name of the EXACT column to return (called "output column")

# 2. Process:
#    - Find the best matching ROW using semantic similarity and fuzzy matching
#    - ONLY after finding the matching row, extract the value from the SPECIFIED output column
#    - Return ONLY that single value - no explanations, no other text

# 3. Output format: 
#    - Return ONLY the raw text value from the specified output column
#    - No formatting, no additional text, no quotes, no markdown

# 4. If no match is found, return the original query text unchanged

# ## VERIFICATION STEP:
# Before responding, verify that:
# 1. You identified the correct matching row
# 2. You are returning ONLY the value from the EXACT column specified
# 3. Your response contains ONLY that value and nothing else

# **Query Text:** {query_text}
# **Columns in the reference data (CSV):** {all_columns}
# **Output column from reference data (CSV) to be returned:** {output_column}
# **Reference Data (CSV):**
# \"\"\"{formatted_data}\"\"\"
#         """
#         print(prompt)
#         response = spark.sql(
#             f"""
#         SELECT ai_query('lakefusion-{endpoint_name}', '{prompt.replace("'", "''")}') AS response
#         """
#         ).collect()[0]["response"]

#         return response
#     except Exception as e:
#         raise Exception(f"Error in value mapper transformation: {str(e)}")

# def f_llm_lookup_execute(
#     input_df: DataFrame,
#     refernce_table: str,
#     input_reference_columns: list,
#     output_reference_columns: str,
#     endpoint_name: str,
#     lookup_column: str,
#     new_column_name: str,
# ):
#     result_df = []
#     list_columns = input_df.columns
    
#     for row in input_df.collect():
#         row_dict = row.asDict()
#         row_values = [row_dict[col] for col in list_columns]
#         query_val = row_dict.get(lookup_column, "")
#         matched = f_llm_lookup(
#             table_path=refernce_table,
#             input_reference_columns=input_reference_columns,
#             output_reference_columns=output_reference_columns,
#             endpoint_name=endpoint_name,
#             query_text=query_val,
#         )

#         result_df.append(row_values + [matched])
#     final_df = spark.createDataFrame(result_df, schema=list_columns + [new_column_name])
#     return final_df

In [0]:
# Transformation handlers
class TransformationHandler:
    @staticmethod
    def handle_null_values(df, config: Dict[str, Any]):
        """
        Handle null values in specified column with static value
        Config format:
        {
            "column": "column_name",
            "replace_type": "static",
            "value": "replacement_value"
        }
        """
        try:
            # Validate configuration
            #TransformationHandler.validate_null_handler_config(config)
            
            # Apply transformation
            return df.fillna(config["value"], subset=[config["column"]])
            
        except Exception as e:
            raise Exception(f"Error in null handler transformation: {str(e)}")

    @staticmethod
    def concat_columns(df, config: Dict[str, Any]):
        """
        Concatenate two columns with a separator
        Config format:
        {
            "column1": "first_column",
            "column2": "second_column",
            "separator": " ",
            "output_column": "concatenated_column"
        }
        """
        try:
            # Validate configuration
            #TransformationHandler.validate_concat_columns_config(config)
            
            # Apply transformation
            return df.withColumn(
                config["output_column"],
                concat(
                    col(config["column1"]),
                    lit(config["separator"]),
                    col(config["column2"])
                )
            )
        except Exception as e:
            raise Exception(f"Error in column concatenation transformation: {str(e)}")

    

    @staticmethod
    def string_cleaner(df, config: Dict[str, Any]):
        """
        Clean string values in specified column
        Config format:
        {
            "column": "column_name",
            "operations": ["trim", "lower", ...],
            "custom_pattern": "regex_pattern",
            "output_column": "output_column_name"
        }
        """
        try:
            # Validate configuration
            #TransformationHandler.validate_string_cleaner_config(config)
            
            # Start with the original column
            result = col(config["column"])
            
            # Apply operations in sequence
            for operation in config["operations"]:
                if operation == "trim":
                    result = trim(result)
                elif operation == "lower":
                    result = lower(result)
                elif operation == "upper":
                    result = upper(result)
                elif operation == "remove_special":
                    result = regexp_replace(result, "[^A-Za-z0-9\\s]", "")
                elif operation == "remove_numbers":
                    result = regexp_replace(result, "\\d", "")
                elif operation == "custom_pattern":
                    result = regexp_replace(result, config["custom_pattern"], "")
            
            # Create new column with cleaned values
            return df.withColumn(config["output_column"], result)
                
        except Exception as e:
            raise Exception(f"Error in string cleaner transformation: {str(e)}")
    
    @staticmethod
    def filter_data(df, config: Dict[str, Any]):
        """
        Filter data based on conditions
        Config format:
        {
            "column": "column_name",
            "operator": "equals",
            "value": "filter_value"
        }
        """
        try:
            # Validate configuration
            #TransformationHandler.validate_filter_config(config)
            
            column = config["column"]
            operator = config["operator"]
            value = config["value"]
            
            # Apply filter based on operator
            if operator == "equals":
                return df.filter(col(column) == value)
            elif operator == "not_equals":
                return df.filter(col(column) != value)
            elif operator == "contains":
                return df.filter(col(column).contains(value))
            elif operator == "starts_with":
                return df.filter(col(column).startswith(value))
            elif operator == "ends_with":
                return df.filter(col(column).endswith(value))
            elif operator == "greater_than":
                # Try to cast value to numeric for comparison
                try:
                    numeric_value = float(value)
                    return df.filter(col(column) > numeric_value)
                except ValueError:
                    raise ValueError(f"Value '{value}' cannot be converted to number for greater_than comparison")
            elif operator == "less_than":
                # Try to cast value to numeric for comparison
                try:
                    numeric_value = float(value)
                    return df.filter(col(column) < numeric_value)
                except ValueError:
                    raise ValueError(f"Value '{value}' cannot be converted to number for less_than comparison")
            else:
                raise ValueError(f"Unsupported operator: {operator}")
                
        except Exception as e:
            raise Exception(f"Error in filter transformation: {str(e)}")

    @staticmethod
    def value_mapper(df, config: Dict[str, Any]):
        """
        Map values in a column based on provided mapping dictionary
        Config format:
        {
            "column": "column_name",
            "mappings": {
                "old_value1": "new_value1",
                "old_value2": "new_value2"
            },
            "default_value": "default",
            "case_sensitive": false
        }
        """
        try:
            # Validate configuration
            #TransformationHandler.validate_value_mapper_config(config)
            
            column = config["column"]
            mappings = config["mappings"]
            default_value = config["default_value"]
            case_sensitive = config.get("case_sensitive", False)
            
            # Create the mapping expression
            if case_sensitive:
                # Direct mapping with case sensitivity
                mapping_expr = create_map([lit(x) for x in chain(*mappings.items())])
                return df.withColumn(
                    column,
                    coalesce(mapping_expr[col(column)], 
                            when(col(column).isNull(), None)
                            .otherwise(lit(default_value)))
                )
            else:
                # Case-insensitive mapping using lower() function
                mapping_expr = create_map([lit(x.lower()) if i % 2 == 0 else lit(y) 
                                        for x, y in mappings.items() 
                                        for i, v in enumerate([x, y])])
                return df.withColumn(
                    column,
                    coalesce(mapping_expr[lower(col(column))], 
                            when(col(column).isNull(), None)
                            .otherwise(lit(default_value)))
                )
                
        except Exception as e:
            raise Exception(f"Error in value mapper transformation: {str(e)}")

    @staticmethod
    def lookup_table(df, config: Dict[str, Any]):
        try:
            llm_endpoint=config['llm_model']
            reference_table=config['reference_table']
            lookup_col=config['column']
            new_column=config['output_column']
            input_reference_columns=config['input_ref_columns']
            output_reference_columns=config['output_ref_column']
           
            
            return f_llm_lookup_execute(input_df=df,
                refernce_table=reference_table,
                input_reference_columns=input_reference_columns,
                output_reference_columns=output_reference_columns,
                endpoint_name=llm_endpoint,
                lookup_column=lookup_col,
                new_column_name=new_column)


        except Exception as e:
            raise Exception(f"Error in lookup  transformation: {str(e)}")

    







In [0]:
# Transformation handlers
class TransformationHandler_Validation:
    @staticmethod
    def validate_null_handler_config(config: Dict[str, Any]):
        required_keys = ["column", "replace_type", "value"]
        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing required key '{key}' in null handler configuration")
        
        if config["column"] not in current_df.columns:
            raise ValueError(f"Column '{config['column']}' not found in input table")
        
        if config["replace_type"] != "static":
            raise ValueError(f"Unsupported replace_type: {config['replace_type']}")

    @staticmethod
    def validate_concat_columns_config(config: Dict[str, Any]):
        required_keys = ["column1", "column2", "separator", "output_column"]
        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing required key '{key}' in concat columns configuration")
        
        if config["column1"] not in current_df.columns:
            raise ValueError(f"Column '{config['column1']}' not found in input table")
        
        if config["column2"] not in current_df.columns:
            raise ValueError(f"Column '{config['column2']}' not found in input table")

    @staticmethod
    def validate_string_cleaner_config(config: Dict[str, Any]):
        required_keys = ["column", "operations", "output_column"]
        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing required key '{key}' in string cleaner configuration")
        
        if config["column"] not in current_df.columns:
            raise ValueError(f"Column '{config['column']}' not found in input table")
        
        if not isinstance(config["operations"], list):
            raise ValueError("Operations must be a list")
        
        valid_operations = ["trim", "lower", "upper", "remove_special", "remove_numbers", "custom_pattern"]
        for op in config["operations"]:
            if op not in valid_operations:
                raise ValueError(f"Invalid operation: {op}")
                
        if "custom_pattern" in config["operations"] and not config.get("custom_pattern"):
            raise ValueError("Custom pattern is required when using custom_pattern operation")

    @staticmethod
    def validate_filter_config(config: Dict[str, Any]):
        required_keys = ["column", "operator", "value"]
        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing required key '{key}' in filter configuration")
        
        if config["column"] not in current_df.columns:
            raise ValueError(f"Column '{config['column']}' not found in input table")
        
        valid_operators = ["equals", "not_equals", "contains", "starts_with", "ends_with", "greater_than", "less_than"]
        if config["operator"] not in valid_operators:
            raise ValueError(f"Invalid operator: {config['operator']}")

    @staticmethod
    def validate_value_mapper_config(config: Dict[str, Any]):
        required_keys = ["column", "mappings", "default_value"]
        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing required key '{key}' in value mapper configuration")
        
        if config["column"] not in current_df.columns:
            raise ValueError(f"Column '{config['column']}' not found in input table")
        
        if not isinstance(config["mappings"], dict):
            raise ValueError("Mappings must be a dictionary")
        
    @staticmethod
    def validate_lookup_config(config: Dict[str, Any]):
        required_keys = ["column", "output_column"]
        
        
        for key in required_keys:
            if key not in config:
                raise ValueError(f"Missing required key '{key}' in value mapper configuration")
        
        if config["column"] not in current_df.columns:
            raise ValueError(f"Column '{config['column']}' not found in input table")
        
       
            

